# 📚  Exercise Session - Week 6, Part 1
**Main Topics**: Generation, Decoding, Beam-search, Sampling, NLG Eval Metrics


Welcome to Week 6 of CS-552 Modern NLP's exercise sessions!
The aim of this exercise is to get yourself acquainted with generation algorithms that are used with language models.
We demonstrate the different generation algorithms with a fine-tuned version of the [GPT-2](https://huggingface.co/openai-community/gpt2) model. You will learn how to generate text using Hugging Face's `generate` method and learn about the limitations of different text generation algorithms.

1. [Story Generator Model:](#1-story-generator-model)
    - Load a GPT-2 model that's only been pretrained
    - Load a GPT-2 model finetuned on the BookCorpus dataset
2. [Decoding Algorithms](#2-decoding-algorithms)
    1. [Greedy Search / Greedy Decoding](#21-greedy-search)
    2. [Beam Search](#22-beam-search)
    3. [Decoding Takeaways](#23-decoding-takeaways)
3. [Sampling Algorithms:](#3-nucleus-sampling)
    1. [Random Sampling](#31-random-sampling)
    2. [Top-$p$ Sampling](#32-top--nucleus)
    3. [Sampling Takeaways](#33-sampling-takeaways)
4. [Automatic NLG Evaluation Metrics - the curious case of BLEU:](#4-automatic-nlg-evaluation-metrics---the-curious-case-of-bleu)
    1. [Pitfall 1: Correct Paraphrases](#41-pitfall-1-accurate-paraphrases)
    2. [Pitfall 2: Senseless Repetitions](#42-pitfall-2-senseless-repetitions)
    3. [Other Problems](#43-other-problems-with-bleu)
5. [Conclusion](#5-conclusion)

> **By the end of the session you will be able to:**
> - ✅  Generate stories with Language Models
> - ✅  Use decoder algorithms such as greedy and beam search with huggingface and understand their pitfalls
> - ✅  Use sampling algorithms such as top-k and top-k sampling with huggingface and understand their pitfalls
> - ✅  Understand the BLEU NLG evaluation metric and recognize its pitfalls

<div style="padding:8px 0 8px 15px;border-left:3px solid orange;">

### ⚠️ **Note: Cells are pre-run. You don't need to execute any cells, but you are welcome to. It will download the finetuned GPT2 model which might take around 1.5-3 minutes depending on your internet connection.** ⚠️
So if you would like to, you can skip this setup section! Otherwise...

### Setup
- Colab: If you are using Colab then download the following packages by creating a code cell below:
    ```shell
    !python -V
    !pip install transformers evaluate
    ```

- Local: If you are using a local setup, then make sure you have a python environment create for Python 3.10 (you can use the same one from Week 1). For example you can use the following command to create a conda environment if you haven't done so for previous exercises:

    ```shell
    conda create -n="mnlp-venv" python=3.10.12
    conda activate mnlp-venv
    ```

    And download the requirements:
    ```
    pip install -r requirements.txt
    ```

    And then make sure to set the kernel of this notebook to that environment, such as when using VS Code.

### Acknowledgements
We are very grateful to [Patrick von Platen](https://huggingface.co/patrickvonplaten) for writing a [blogpost on different generation techniques](https://huggingface.co/blog/how-to-generate). Our notebook takes its examples, images, and wordings.
Moreover, we would like to thank [Google Cloud Translation's evaluation documentation](https://docs.cloud.google.com/translate/docs/bleu-scores), which we have used for the evaluation section. We encourage you to check these resources out!

The model we primarily use is the 125M parameter [GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf). The [finetuned story generation version on huggingface](https://huggingface.co/pranavpsv/genre-story-generator-v2) was finetuned using the [BookCorpus dataset](https://paperswithcode.com/dataset/bookcorpus) by getting the different genres per book and prefixing the inputs with the relevant genre.

</div>


---

## 1) Story Generator Model

First, we import the pretrained GPT-2 and a finetuned version of it with [huggingface](https://huggingface.co/)'s [transformers package](https://huggingface.co/docs/transformers/index). The latter is fine-tuned on sample of BookCorpus dataset for short story generation. This model has been fine-tuned to first take the `<BOS>` token and then a token that depicts the genre we desire to generate such as `<adventure>`. Possible genres this model handles and their respective tokens are:

- Romance `<romance>`
- Adventure `<adventure>`
- Mystery & detective `<mystery-&-detective>`
- Fantasy `<fantasy>`
- Humor & comedy `<humor-&-comedy>`
- Paranormal `<paranormal>`
- Science fiction `<science-fiction>`

In [3]:
## Set up the device
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available() and torch.backends.mps.is_built(): # enable the usage of Apple silicon
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Using device on: [{device}]")

Using device on: [cpu]


In [ ]:
import torch
import numpy as np
import random
import os

def set_random_seed(seed):
    """
    Set random seed for model training or inference.

    Args:
        seed (int): defines which seed to use.
    """
    # fix random seeds for reproducibility
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    # benchmark=True works faster but reproducibility decreases
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

In [ ]:
# 1) Import the relevant packages
import torch
import evaluate
from transformers import GPT2LMHeadModel, GPT2Tokenizer


# 2) Load the relevant models and their respective tokenizers
# NOTE: You could also try gpt2-medium for the pretrained model! Although generation will be a bit slower.
pretrained_model_id = "gpt2"
story_model_id = "pranavpsv/genre-story-generator-v2"
#
pretrained_tokenizer = GPT2Tokenizer.from_pretrained(pretrained_model_id)
story_tokenizer = GPT2Tokenizer.from_pretrained(story_model_id)
#
pretrained_model = GPT2LMHeadModel.from_pretrained(
    pretrained_model_id, pad_token_id=pretrained_tokenizer.eos_token_id
)
story_model = GPT2LMHeadModel.from_pretrained(
    story_model_id, pad_token_id=story_tokenizer.eos_token_id
)

# Move the models to the device
pretrained_model.to(device)
story_model.to(device)
#
pretrained_model.eval()
story_model.eval()


# 3) A display function to compare the pretrained and story-finetuned GPT-2 model generations
def display_outputs(pretrained_output, story_output, multiple_output=False):
    if not multiple_output:
        print(80 * "-")
        print("Pretrained model output:")
        print(80 * "-")
        print(
            pretrained_tokenizer.decode(pretrained_output[0], skip_special_tokens=True)
        )
        print(80 * "-")
        print("Story model output:")
        print(80 * "-")
        print(story_tokenizer.decode(story_output[0], skip_special_tokens=True))
    else:
        print(80 * "-")
        print("Pretrained model output:")
        print(80 * "-")
        for i, beam_output in enumerate(pretrained_output):
            if i > 0:
                print(20 * "-")
            print(
                "{}: {}".format(
                    i + 1,
                    pretrained_tokenizer.decode(beam_output, skip_special_tokens=True),
                )
            )

        print(80 * "-")
        print("Story model output:")
        print(80 * "-")
        for i, beam_output in enumerate(story_output):
            if i > 0:
                print(20 * "-")
            print(
                "{}: {}".format(
                    i + 1, story_tokenizer.decode(beam_output, skip_special_tokens=True)
                )
            )

## 2) Decoding Algorithms

### 2.1) Greedy Search / Greedy Decoding

Greedy search is the first type of decoding algorithm one can imagine: we select the next token as the one with the highest probability.
Given the following image provided by Patrick von Platen, the path that a greedy search would choose is depicted by the red path.

<!-- ![Greedy Search](./figs/greedy_search.png) -->
<!-- <center><img width="500" src="figs/greedy_search.png"/></center> -->
<center><img width="500" src="https://i.ibb.co/23bvTHtq/greedy-search.png"/></center>


Given the word **The**, the algorithm picks *greedily* the highest probability token **nice**, and then **woman**. Note that the generated token sequence has therefore a probability of:

$0.5 \times 0.4 = 0.2$.

Now let's generate a sequence after inputting the model the sequence **"Bella couldn't take it anymore. Edward was gone for 3 months now and"** inspired by the second movie and book of Twilight. For the story generation version we will add the prefix token `<romance>`.

<center><img width="400" src="https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExaHc5cGV6bGhtNHNteGlqcTg2Mzd0YXJvY2J5aHFlb2JtbHlteHFzZiZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/4LsTIyZSnwS0bPz2gf/giphy.gif"/></center>

Note that in `transformers`, we can generate with autoregressive models simply by calling the model's `generate` function and giving it tokenized inputs as shown below:

In [6]:
# Tokenize the text with which we condition the generation
pretrained_input = pretrained_tokenizer(
    "Bella couldn't take it anymore. Edward was gone for 3 months now and",
    return_tensors="pt"
)

story_input = story_tokenizer(
    "<romance> Bella couldn't take it anymore. Edward was gone for 3 months now and",
    return_tensors="pt"
)

# Move the input tensors to the device
pretrained_input = {key: value.to(device) for key, value in pretrained_input.items()}
story_input = {key: value.to(device) for key, value in story_input.items()}

pretrained_input_ids = pretrained_input["input_ids"]
pretrained_input_att_mask = pretrained_input["attention_mask"]
story_input_ids = story_input["input_ids"]
story_input_att_mask = story_input["attention_mask"]

In [7]:
# Generate text until the output length (which includes the context length)
#    reaches max_length
greedy_pretrained_output = pretrained_model.generate(
    pretrained_input_ids,
    attention_mask=pretrained_input_att_mask,
    max_length=100
)

greedy_story_output = story_model.generate(
    story_input_ids,
    attention_mask=story_input_att_mask,
    max_length=100
)

# 3) Decode the output back into readable strings
display_outputs(greedy_pretrained_output, greedy_story_output)

--------------------------------------------------------------------------------
Pretrained model output:
--------------------------------------------------------------------------------
Bella couldn't take it anymore. Edward was gone for 3 months now and Bella was still in the hospital.

"I'm so sorry, Bella," Edward said. "I'm so sorry. I'm so sorry. I'm so sorry. I'm so sorry. I'm so sorry. I'm so sorry. I'm so sorry. I'm so sorry. I'm so sorry. I'm so sorry. I'm so sorry. I'm so sorry. I'm so
--------------------------------------------------------------------------------
Story model output:
--------------------------------------------------------------------------------
<romance> Bella couldn't take it anymore. Edward was gone for 3 months now and Bella was still in love with him. Bella wanted to be with Edward but she couldn't. Bella wanted to be with Edward but she couldn't. Bella wanted to be with Edward but she couldn't. Bella wanted to be with Edward but she couldn't. Bella w

**Question:**
- **Given the generated output above, what is a serious problem you find about the greedy search algorithm?**
- **Why do you think this might be happening?**
- **Is there a difference between the pretrained model and the story finetuned model with respect to these pitfalls?**

*Answer: TODO - your answer here!*

*Answer Solution:*

- The greedy algorithm repeats a lots of subsequences for both the pretrained and story models. In the former, the repetition that gets amplified is "I'm so sorry". In the latter it's "Bella wanted to be with Edward but she couldn't."

- Language models assign higher probabilities to sequences with repetitions. As you saw in the lectures, the negative log likelihood (NLL) of a repetitive sequence keeps on decreasing with its length. This is a consequence of minimizing the NLL objective and training data, which together promote copy-like behavior. (For more details, you can check out these papers: [1](https://arxiv.org/abs/1904.09751), [2](https://arxiv.org/abs/2206.02369), [3](https://arxiv.org/abs/2012.14660)). Therefore, any algorithm that maximizes likelihood, either more locally (greedy decoding) or more globally (beam search) is bound to get repetitive generations.

- There isn't a significant difference. Both suffer the same problem of repetitions. We can note though that the repetition of the story model is less generic and more grounded to the story.

### 2.2) Beam Search

Instead of considering a single best next token, the beam search algorithm tries to respond to problems with greedy search by considering the next `num_beams` amount of tokens that have the highest joint probability. Consider the initial token sequence probability tree illustration with `num_beams=2` (since `num_beams=1` would be greedy decoding!):

<!-- ![Beam Search](./figs/beam_search.png) -->
<!-- <center><img width="500" src="figs/beam_search.png"/></center> -->
<center><img width="500" src="https://i.ibb.co/RmZ8FHk/beam-search.png"/></center>

- Assume that **The** is selected at time step = 0.
- At time step 1, beam search calculates the probabilities of all possible continuations and chooses the top-`num_beams` ones. Hence, besides the most likely hypothesis **The nice**, beam search also keeps track of the second most likely one **The dog**.
- At time step 2, beam search again assesses all possible continuations and chooses the top-`num_beams` ones. This is the last step (since we have only three timesteps total) with two candidate sequences: **The dog has** and **The nice woman**. Note that beam search is able to find the word sequence **The dog has** that has a higher overall probability of $0.36$ ($0.4 \times 0.9$) than **The nice woman**, which has a probability of 0.2 ($0.5 \times 0.4$).


$\implies$ by keeping track of more than one most likely sequences at each step, beam search is able to find a sequence that is more likely than what greedy decoding would produce.

**Note**: with `num_beams` set to $1$, the beam search acts just as the greedy decoding. Thus, we can turn the beam search decoding on/off by controlling a single argument `num_beams` with its value set to 1 defining the greedy decoding and any value greater than 1 activating the beam search.

Let's generate what beam search would give for the same input sentence **"Bella couldn't take it anymore. Edward was gone for 3 months now and"**.

We also do `early_stopping=True`, where beam search continues generating tokens until it finds `num_beams` beams that are complete (i.e., ended with [EOS]). It doesn’t wait for `max_length` anymore.

In [8]:
# Generate text using beam search until the output length (which includes the context length)
#    reaches max_length or early stopping
beam_pretrained_output = pretrained_model.generate(
    pretrained_input_ids,
    attention_mask=pretrained_input_att_mask,
    max_length=100,
    num_beams=4,  # NOTE: set num_beams=4, default num_beams=1
    early_stopping=True,
)
beam_story_output = story_model.generate(
    story_input_ids,
    attention_mask=story_input_att_mask,
    max_length=100,
    num_beams=4,  # NOTE: set num_beams=4, default num_beams=1
    early_stopping=True,
)

# Decode the output back into readable strings
display_outputs(beam_pretrained_output, beam_story_output)

--------------------------------------------------------------------------------
Pretrained model output:
--------------------------------------------------------------------------------
Bella couldn't take it anymore. Edward was gone for 3 months now and Bella had no idea what was going on.

"I'm sorry, Bella," Edward said.

"I'm sorry," Bella said.

"I'm sorry," Edward said.

"I'm sorry," Bella said.

"I'm sorry," Edward said.

"I'm sorry," Bella said.

"I'm sorry," Edward said.

"
--------------------------------------------------------------------------------
Story model output:
--------------------------------------------------------------------------------
<romance> Bella couldn't take it anymore. Edward was gone for 3 months now and Bella couldn't take it anymore. Bella wanted to be with Edward, but she couldn't take it anymore. Bella wanted to be with Edward, but she couldn't take it anymore. Bella wanted to be with Edward, but she couldn't take it anymore. Bella wanted to be w

**Question:**
- **Given the generated output above, what is a serious problem about the beam search algorithm?**
- **What is (slightly) different between the greedy search and the beam search outputs?**
- **Can you think of a possible solution to this problem that involves changing the distribution? (even if it's hacky)**

*Answer: TODO - your answer here!*

*Answer Solution:*

- The repetition problem still persists. There is the following difference compared to greedy decoding though.
- The repeated sequences seem to be longer or spanning multiple sentences (*e.g.* Bella said - Edward said - Bella said - Edward said etc.). With greedy decoding, repetitions were shorter spans.
- A possible solution is to force the model to not repeat past sequences by caching them and then setting their probability to 0 according to an allowed repetition rate.


~ Don't look below before answering the question, it has spoilers ;) ~

.

.

.

.

.

.

.

.

.

.

.

.

.

A possible solution proposed by [Paulus et al. (2017)](https://arxiv.org/abs/1705.04304) and [Klein et al. (2017)](https://arxiv.org/abs/1701.02810) is to penalize the model for generating a sequence of grams. ["gram" is just another term for "word". $n$-gram means $n$ consecutive grams.]

One way is to limit how often we allow an $n$-gram to occur by manually setting the probability of next words that could create an already seen $n$-gram to 0.

Hugging Face provides `no_repeat_ngram_size` parameter (see the [documentation](https://huggingface.co/docs/transformers/v4.49.0/en/main_classes/text_generation#transformers.GenerationConfig.no_repeat_ngram_size)), which when set to $k$, limits the occurrences of $k$-grams to one. 

Let's try it out by setting this parameter to 2, such that no $2$-gram appears more than once.

In [9]:
# Generate text using beam search with repeating n-gram penalty until the output length (which includes the context length)
#    reaches max_length or early stopping
beam_pretrained_output = pretrained_model.generate(
    pretrained_input_ids,
    attention_mask=pretrained_input_att_mask,
    max_length=100,
    num_beams=4,
    no_repeat_ngram_size=2,  # NOTE: set no_repeat_ngram_size=2
    early_stopping=True,
)
beam_story_output = story_model.generate(
    story_input_ids,
    attention_mask=story_input_att_mask,
    max_length=100,
    num_beams=4,
    no_repeat_ngram_size=2,  # NOTE: set no_repeat_ngram_size=2
    early_stopping=True,
)

# 3) Decode the output back into readable strings
display_outputs(beam_pretrained_output, beam_story_output)

--------------------------------------------------------------------------------
Pretrained model output:
--------------------------------------------------------------------------------
Bella couldn't take it anymore. Edward was gone for 3 months now and Bella had no idea what was going on.

"I don't know what to do," Bella said. "I just want to go back to school. I'm not going to be able to take care of my family anymore."
--------------------------------------------------------------------------------
Story model output:
--------------------------------------------------------------------------------
<romance> Bella couldn't take it anymore. Edward was gone for 3 months now and Bella wanted to go back to her old life with Edward. Bella decided to leave Edward and move back in with her aunt and uncle. But Bella's aunt was not happy with the way Bella was living. She wanted Bella to stay in the house, but Bella didn't want to be a part of the family.  Bella and Edward had a lot of fun

Yay! It looks like most problematic repetitiveness is avoided. However, note that this an extreme approach.

**Question:**
- **What could be a problem with this penalization approach? (Hint: think about proper nouns)**
- **There is also a `repetition_penalty` argument in generate, that instead of penalizing $n$-grams penalizes every token that’s repeating. What could be a problem with this argument? (Hint: think about how the GPT-2 tokenizer works)**

    > repetition_penalty (float, optional, defaults to 1.0) — The parameter for repetition penalty. 1.0 means no penalty. See [this paper](https://arxiv.org/pdf/1909.05858.pdf) for more details.

*Answer: TODO - your answer here!*

*Answer Solution:*
- What if some 2-grams for named entities (*e.g.* São Paulo or New York) need to be repeated because the article is about that entity and we penalized them? The decoding algorithm won't re-use the bigram, and the text could look very unnatural.
- In the case of GPT-2, the BPE tokenizer considers subwords as tokens, such as the middle or end of a word, stopwords, and punctuation. If the penalty is high then it could demote something as simple as a plurality indicating token "s" after a word like "table(s)". It may lead to some ungrammatical/unnatural sentences.


Another important property of generation is *variety*. For an application such as a writing aid, we might want to see different generated sentences, and pick the one we like the most. Given that there could be some subsequences that have close probabilities, can we get different outputs from beam-search? We use `num_return_sequences` for this:

In [10]:
# Generate text using beam search with n-gram penalty and returning multiple sequences
# until the output length (which includes the context length)
#    reaches max_length or early stopping
beam_pretrained_output = pretrained_model.generate(
    pretrained_input_ids,
    attention_mask=pretrained_input_att_mask,
    max_length=100,
    num_beams=4,
    no_repeat_ngram_size=2,
    early_stopping=True,
    num_return_sequences=3,  # NOTE: set num_return_sequences = 3
)
beam_story_output = story_model.generate(
    story_input_ids,
    attention_mask=story_input_att_mask,
    max_length=100,
    num_beams=4,
    no_repeat_ngram_size=2,
    early_stopping=True,
    num_return_sequences=3,  # NOTE: set num_return_sequences = 3
)

# 3) Decode the output back into readable strings
display_outputs(beam_pretrained_output, beam_story_output, multiple_output=True)

--------------------------------------------------------------------------------
Pretrained model output:
--------------------------------------------------------------------------------
1: Bella couldn't take it anymore. Edward was gone for 3 months now and Bella had no idea what was going on.

"I don't know what to do," Bella said. "I just want to go back to school. I'm not going to be able to take care of my family anymore."
--------------------
2: Bella couldn't take it anymore. Edward was gone for 3 months now and Bella had no idea what was going on.

"I don't know what to do," Bella said. "I just want to go back to school. I'm not going to be able to take care of my family. It's been a long time since I've been here, and I can't do anything about it. But I know that if I do, I'll be back in the world.
--------------------
3: Bella couldn't take it anymore. Edward was gone for 3 months now and Bella had no idea what was going on.

"I don't know what to do," Bella said. "I just wan

**Question:**
- **What is the variation mostly related to?**
- **Given the results do you think this algorithm can be applied as a writing aid?**

*Answer: TODO - your answer here!*

*Answer Solution:*

- The source of the variation is hard to spot as it's mainly where the sentence stops or the type of punctuation used. The outputs can be mostly described as "predictable" rather than "varied".

- No, if the goal of this algorithm was to write stories, then it wouldn't generate many different stories. It could be useful as a summarization tool, as the output is more predictable.

Note that changing `num_beams` may introduce qualitative changes in some cases while also affecting the generation speed. Feel free to try different `num_beams` options and compare how the generated sequences differ and how slower the generation becomes.

### 2.3) Decoding Takeaways

As you can see for most generations, the most likely subsequence is unique and decoding algorithms can be too predictable. This is mainly because of two reasons:

- **Generation length:** Beam search works when tasks have predictable length (ex: machine translation or summarization) ([Murray et al. 2018](https://arxiv.org/abs/1808.10006), [Yang et al. 2018](https://arxiv.org/abs/1808.09582)). It's not great for open-ended generation where the desired output length can vary (ex: dialog).

- **Suprisal effect:** [Ari Holtzman et al. (2019)](https://arxiv.org/abs/1904.09751) show that high quality human language does not follow a distribution of high probability next words. It's desirable for the generated text to be surprising.

Therefore, we need to consider stochastic generation algorithms. Will Bella ever get over Edward? Let's sample to see what happens!

## 3) Sampling Algorithms

### 3.1) Random Sampling

Sampling means randomly picking the next word $w_t$ from ​its conditional probability distribution $P(w_{t}|w_{1:t-1})$. Let's picture what this can be like with the prior probability trees we have seen:

<!-- ![Beam Search](./figs/sampling_search.png) -->
<!-- <center><img width=800" src="figs/sampling_search.png"/></center> -->
<center><img width="800" src="https://i.ibb.co/vx9Ck8JB/sampling-search.png"/></center>


Each dot here represents a timestamp in decoding. To choose the second token **car** we randomly pick a token among [**nice**, **dog**, **car**]. The model then gives next possible tokens' probabilities and the generation algorithm *randomly* chooses **drives**, which happens to be the top-probability token.

However, as you can imagine, this may be *too* unpredictable. Let's try it out with GPT-2!

In [11]:
# Generate text using random sampling until the output length (which includes the context length)
#    reaches max_length or early stopping

# NOTE: setting the seed so the output doesn't always change at every run,
#       but you can comment the next line out to see how much it can vary.

seed = 84
set_random_seed(seed)
sample_pretrained_output = pretrained_model.generate(
    pretrained_input_ids,
    attention_mask=pretrained_input_att_mask,
    do_sample=True, max_length=100, top_k=0
)
set_random_seed(seed)
sample_story_output = story_model.generate(
    story_input_ids,
    attention_mask=story_input_att_mask,
    do_sample=True, max_length=100, top_k=0
)

# 3) Decode the output back into readable strings
display_outputs(sample_pretrained_output, sample_story_output)

--------------------------------------------------------------------------------
Pretrained model output:
--------------------------------------------------------------------------------
Bella couldn't take it anymore. Edward was gone for 3 months now and she'd once again turned into a ghost.

Jessia tried to escape. Things had changed. Her vocation was free. She tried to stay there but died rather than spend time alone on the tired edge of her own life. Nothing. But now she had to, even though she thought it came down to the nostalgia she had for her old life.

When there was a problem she wouldn't have
--------------------------------------------------------------------------------
Story model output:
--------------------------------------------------------------------------------
<romance> Bella couldn't take it anymore. Edward was gone for 3 months now and she was seeing a kind of dark older man with aged eyes. Bella don't understand him anymore. Instead of calling out, she picks u

**Question:**
- **What do you observe about the output? What is particularly wrong about it? What is the price we pay for variation?**
- **What is the primary source for this problem?**
- **What are some ways to deal with this problem?**

*Answer: TODO - your answer here!*

*Answer Solution:*

- As expected the model repeats subsequences much less, but the output is extremely incoherent and sometimes ungrammatical (Bella don't understand him anymore.).
- This is due to the way we are randomly sampling from the **complete** output distribution. Even if the probability of sampling the least-likely token is extremely low (think of a vocabulary size of ~50K), we are still allowing the sampling algorithm to sample from the extreme tail of the distribution. This can lead to many inconsistencies in the way the text is generated and reduce overall coherence.
- One possibility is to limit somehow the tail end of the distribution (e.g. top-$k$ or top-$p$ algorithms!).

### 3.2) Top-$p$ (nucleus) and top-k


<!-- ![Top-P Sampling](./figs/top_p_sampling.png) -->
<!-- <center><img width=800" src="figs/top_p_sampling.png"/></center> -->
<center><img width="800" src="https://i.ibb.co/B5D4Py5n/top-p-sampling.png"/></center>


In class, we went over top-$p$ sampling (or also commonly referred to as nucleus sampling). The "thresholding" aspect of this algorithm is the cumulative probability mass $p$ that limits the top probability tokens that we can sample from. This way the number of best $k$ tokens dynamically changes according to the model's confidence. Let's see how we can implement it with the `transformers` package. There are 2 types of arguments that we can pass to the model's `generate` function:
1. `top_k` : The top $k$ highest probability tokens kept for sampling
2. `top_p` : The cumulative probability mass from which the most likely tokens are sampled. If set to float < 1, only the smallest set of most probable tokens with probabilities that add up to $p$ **or higher** are kept for sampling.
    - **Or higher** because if no single token falls under this cumulative probability mass, for example, one token has a probability mass higher than `top_p`, then the default in huggingface is to greedily sample the top-scoring token.

We set `top_k=0` to only demonstrate nucleus sampling with `top_p=0.7`.

In [9]:
# Generate text using top-p (nucleus) sampling until the output length (which includes the context length)
#    reaches max_length

# NOTE: setting the seed so the output doesn't always change at every run,
#       but you can comment the next line out to see how much it can vary.

seed = 84
set_random_seed(seed)
sample_pretrained_output = pretrained_model.generate(
    pretrained_input_ids,
    attention_mask=pretrained_input_att_mask,
    do_sample=True,
    max_length=100,
    top_p=0.7,  # NOTE: sample only from tokens falling in the top cumulative 70% probability
    top_k=0,  # NOTE: deactivate top_k sampling
)
set_random_seed(seed)
sample_story_output = story_model.generate(
    story_input_ids,
    attention_mask=story_input_att_mask,
    do_sample=True,
    max_length=100,
    top_p=0.7,  # NOTE: sample only from tokens falling in the top cumulative 70% probability
    top_k=0,  # NOTE: deactivate top_k sampling
)

# 3) Decode the output back into readable strings
display_outputs(sample_pretrained_output, sample_story_output)

--------------------------------------------------------------------------------
Pretrained model output:
--------------------------------------------------------------------------------
Bella couldn't take it anymore. Edward was gone for 3 months now and she'd lost a lot of her feelings.

He'd tried to reassure her by saying that he'd had a few encounters with the Ruby, but that was just a tip of the iceberg.

"I don't know how you'd handle this, Edward," she said, tilting her head to look at him. "It's really like a nightmare, Edward."

"It's really like
--------------------------------------------------------------------------------
Story model output:
--------------------------------------------------------------------------------
<romance> Bella couldn't take it anymore. Edward was gone for 3 months now and she was seeing a new guy. Bella wanted to be with him but she couldn't because she was afraid of him. When Bella is down with her new boyfriend Simon, her mother decides to get

The coherence is somewhat better with respect to the context given! But what about variance? Let's see by sampling multiple sequences.

In [10]:
# Generate text using top-p (nucleus) sampling until the output length (which includes the context length)
#    reaches max_length; sample multiple sequences
# NOTE: setting the seed so the output doesn't always change at every run,
#       but you can comment the next line out to see how much it can vary.
seed = 84
set_random_seed(seed)
sample_pretrained_output = pretrained_model.generate(
    pretrained_input_ids,
    attention_mask=pretrained_input_att_mask,
    do_sample=True,
    max_length=100,
    top_p=0.7,  # sample only from tokens falling in the top cumulative 70% probability
    top_k=0,  # deactivate top_k sampling
    num_return_sequences=3,  # set num_return_sequences = 3
)
set_random_seed(seed)
sample_story_output = story_model.generate(
    story_input_ids,
    attention_mask=story_input_att_mask,
    do_sample=True,
    max_length=100,
    top_p=0.7,  # sample only from tokens falling in the top cumulative 70% probability
    top_k=0,  # deactivate top_k sampling
    num_return_sequences=3,  # set num_return_sequences = 3
)

# 3) Decode the output back into readable strings
display_outputs(sample_pretrained_output, sample_story_output, multiple_output=True)

--------------------------------------------------------------------------------
Pretrained model output:
--------------------------------------------------------------------------------
1: Bella couldn't take it anymore. Edward was gone for 3 months now and she'd lost a lot of her feelings.

He'd tried to reassure her by saying that he'd had a few encounters with the Ruby, but that was just a tip of the iceberg.

"I don't know how you'd handle this, Edward," she said, tilting her head to look at him. "It's really like a nightmare, Edward."

"It's really like
--------------------
2: Bella couldn't take it anymore. Edward was gone for 3 months now and he wanted to know how he was going to do things for her.

So she asked her dad for his help in finding Edward. They agreed and he told Bella how to keep her safe.

He told her how to get more money and help them make sure Edward didn't come back. He told her he was sorry and he had to go. He told her he wanted to get to the most valuable
-

As we somewhat expected, the outputs are much more diverse than those of deterministic decoding algorithms. And it looks like Bella does not get a happy ending if `num_return_sequences=3` 🥲 even if all three endings are unique in their own way. Well, as long as there are no creepy CGI babies!

<center><img width="500" src="https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExZXI4M3NtajI5a3p6MjlpMTJxaWw2d2ExbG94aWszMTE2cTBraGY1ZyZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/kk1m3QsOfZ3tklAs4K/giphy-downsized-large.gif"/></center>


### 3.3) Sampling Takeaways

Sampling can be really finicky according to the input text we condition our generation with and the top-$k$ or top-$p$ hyperparameter options we choose.

However, it offers a wider variety of outputs rather than decoding algorithms like greedy and beam search.

Note the differences between the story model and the pretrained model generations. A pretrained language model trained with next-token prediction learns general linguistic patterns and can generate plausible text, but it does not inherently specialize in producing structured narratives, and, therefore, the generated sequences are not necessarily story-like without additional prompting. Fine-tuning on story data biases the model toward narrative conventions, making its outputs more consistently story-like and coherent.

## 4) Automatic NLG Evaluation Metrics - the curious case of BLEU

### Motivation
In this section, we will take a close look on how to evaluate generated text. In translation or other sequence-to-sequence tasks, we do not have a single way to get a *correct* answer. For example, consider a model that outputs the text:

> The cat is fluffy.

and the correct *reference* text we have is:

> The cat has fluffy hair.

These two sentences look like paraphases but are not exact matches. Therefore these tasks require a more flexible evaluation in which we check for matching subsequences of $n$-grams. For this reason, we will take a close look to the popular BLEU score (BiLingual Evaluation Understudy).

### What is BLEU?
BLEU is a metric proposed by [Papineni et al (2002)](https://aclanthology.org/P02-1040.pdf) for automatic evaluation of machine translation. The BLEU score is a number between 0 and 1 that compares the similarity of a predicted *candidate* text to a set of correct *reference* texts, where 1 means that the generated text is close to the reference and is of "high quality" and 0 is the opposite. BLEU was originally used for machine translation but can also be applied for any text generation task (e.g., summarization). Note that often in practice the BLEU score is reported on a scale of 1 to 100 for granularity, but it's not an accuracy metric!

So how do we calculate BLEU? In short, it has two terms:

1. **$n$-gram overlap calculation:** BLEU calculates the $n-$gram overlap counts of how many *candidate* unigrams, bigrams, trigrams, and four-grams ($n$=1,...,4) match their $n$-gram counterpart in the *reference* sequence. **This can therefore be considered as a precision metric.** While unigrams might be capturing lexical level similarity, we can imagine that longer $n$-gram matching captures higher-level syntactic constituent similarity.

2. **brevity penalty:** Since BLEU doesn't capture recall with the overlap calculation, we augment it with a brevity penalty. This penalizes generations that are too short compared to a reference length.

For more details and an exact equation, checkout Google Cloud Translation's [evaluation documentation](https://cloud.google.com/translate/automl/docs/evaluate#bleu).

BLEU is an automatic and an easy-to-use metric, but it has many problems depending on where and how it's used. Let's find out what these issues may be!

### 4.1) Pitfall 1: Accurate paraphrases

BLEU measures gram-by-gram similarity between the *candidate* and the *reference* text. This means that, correct paraphrases can hurt the score. Let's see how this might happen when we want to translate:
- input: **le chat s'est assis sur le tapis**
- candidate/prediction/output: **the feline layed over the rug**
- reference: **the cat sat on the mat**

In [12]:
# NOTE: the huggingface evaluate package provides a BLEU score that returns
#       precision at all 4 levels and how much brevity penalty was applied
bleu = evaluate.load("bleu")

# 1) Accurate paraphrase
predictions = ["the feline layed over the rug"]
references = [
    ["the cat sat on the mat"],
]
results = bleu.compute(predictions=predictions, references=references)
results

{'bleu': 0.0,
 'precisions': [0.3333333333333333, 0.0, 0.0, 0.0],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0,
 'translation_length': 6,
 'reference_length': 6}

As you can see while the paraphased sentence is close to the reference, it gets only a slight score on BLEU-1 (leftmost in precisions), and none of BLEU-4 (rightmost in precisions) as there is no four-gram overlap. On the other hand, let's take a look at a semantically incorrect but bag-of-words-wise close sentences.

In [13]:
# 2) Removed stopwords
predictions = ["cat sat mat"]
references = [
    ["the cat sat on the mat"],
]
results = bleu.compute(predictions=predictions, references=references)
print(results)

# 3) switched "mat" and "cat"
predictions = ["the mat sat on the cat"]
references = [
    ["the cat sat on the mat"],
]
results = bleu.compute(predictions=predictions, references=references)
results

{'bleu': 0.0, 'precisions': [1.0, 0.5, 0.0, 0.0], 'brevity_penalty': 0.36787944117144233, 'length_ratio': 0.5, 'translation_length': 3, 'reference_length': 6}


{'bleu': 0.0,
 'precisions': [1.0, 0.8, 0.25, 0.0],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0,
 'translation_length': 6,
 'reference_length': 6}

For the second example, every gram that is in the reference text is also in the prediction. Therefore, BLEU-1 gets full score and BLEU-3 gets a reasonable precision. However, due to the fact that there are no four-gram overlaps, the overall BLEU score is 0.0.

### 4.2) Pitfall 2: Senseless repetitions

Are there any ways we could get the four-gram overlap without coherent mapping among the prediction and the reference?

- input: **le chat s'est assis sur le tapis**
- prediction:
    - **the cat sat on the cat, is on the cat**, or
    - **the cat sat on the cat, the mat sat on the mat**
- reference: **the cat sat on the mat**

In [14]:
predictions = ["the cat sat on the cat, is on the cat"]
references = [
    ["the cat sat on the mat"],
]
results = bleu.compute(predictions=predictions, references=references)
print(results)

predictions = ["the cat sat on the cat, the mat sat on the mat"]
references = [
    ["the cat sat on the mat"],
]
results = bleu.compute(predictions=predictions, references=references)
results

{'bleu': 0.35084396956386854, 'precisions': [0.45454545454545453, 0.4, 0.3333333333333333, 0.25], 'brevity_penalty': 1.0, 'length_ratio': 1.8333333333333333, 'translation_length': 11, 'reference_length': 6}


{'bleu': 0.38058030016749456,
 'precisions': [0.46153846153846156,
  0.4166666666666667,
  0.36363636363636365,
  0.3],
 'brevity_penalty': 1.0,
 'length_ratio': 2.1666666666666665,
 'translation_length': 13,
 'reference_length': 6}

This is the first time we got a non-zero BLEU score. As you can see, none of the translations actually make sense. It's simply because we can find matching four-grams like "the cat sat on" ... "sat on the mat". Despite all 1-grams showing up in the prediction, the BLEU-1 score is reduced because the prediction is long.

### 4.3) Other problems with BLEU

There are some important properties of BLEU that we need to consider when deciding to use it. For example:
1. BLEU performs particularly bad when tested on individual *reference* sentences, or when the length of the generation isn't consistent across a corpus.
2. At least one matching 4-gram is required to get a BLEU score > 0.
3. This score is also very dependent on the sort of tokenization used to get the $n$-grams.

## 5) Conclusion

We found out that greedy and beam search often result in repetitive patterns where the generation loops on subsequences.
However, there are penalization and sampling approaches that help us get coherent generation without the repetitive subsequences.

We also went over a popular NLG evaluation metric and saw how tough it can be to evaluate a task where the output is allowed to be diverse sequences.
While BLEU is one option, there are many more metrics out there. Often in practice we have to use all metrics at the same time to have the big picture.
Otherwise, as we have seen with BLEU, every metric has its own limitations and cannot give us a holistic understanding on how our model is behaving compared to a given large corpus of high quality references.

### What else can I do?

Checkout the huggingface [transformers text generation documentation](https://huggingface.co/docs/transformers/main_classes/text_generation) to see what other methods you can generate with. There are many more sampling approaches being offered these days, and it is a hot topic in NLP. If you want to read a bit more on new methods, checkout this paper on [Locally Typical Sampling](https://arxiv.org/abs/2202.00666) (a generation option that is now available with the *transformers'* `generate` function).

You can also browse through their selection of [metrics](https://huggingface.co/docs/datasets/main/en/how_to_metrics#compute-scores) or the torch package's [metrics](https://torchmetrics.readthedocs.io/en/stable/all-metrics.html#) to find more NLG metrics.